# Transcriptome-wide trans-fit parameter comparison

Compares fitted trans parameters (observed log2FC, EC50, Hill coefficient, ...) between two bayesDREAM runs, gene by gene, for every cis gene both runs have completed a `fit_trans` for.

Unlike `dose_response_comparison.ipynb` (which reloads full models to draw per-gene dose-response curves, and only scales to ~100 genes), this notebook reads each run's `trans_feature_summary_gene.csv` directly -- so it scales to Morris/Replogle's transcriptome-wide trans gene sets. See `comparative/trans_param_compare.py` for the underlying library.

Starts with the simplest comparison (observed log2FC, Morris on x vs Replogle on y), then a grid of other parameters colored + sorted by observed log2FC. Replogle is much less powered than Morris, so points that are `is_dependent` in Morris are outlined -- watch where those land in Replogle's noisier axis.

In [ ]:
# run "pip install ipython-autotime" in your conda env
%load_ext autotime

import os
import sys

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import matplotlib.pyplot as plt
import pandas as pd

from comparative.datasets import DOMINGO, MORRIS, REPLOGLE, ALL_DATASETS
from comparative.trans_param_compare import (
    load_trans_summary,
    merge_pair,
    plot_obs_log2fc,
    plot_param_grid,
    compare_cis_gene,
    compare_all_shared_cis_genes,
    DEFAULT_GRID_PARAMS,
)

## Config

In [ ]:
# Which two datasets to compare, and in which x/y order (matters for the
# scatter plots and for which dataset's is_dependent flag gets highlighted --
# see plot_obs_log2fc/plot_param_grid docstrings, dataset A drives both).
SPEC_X = MORRIS      # x-axis
SPEC_Y = REPLOGLE    # y-axis

PLOT_DIR = './trans_param_comparison_plots'
os.makedirs(PLOT_DIR, exist_ok=True)

# Extra parameters for the grid plot, beyond DEFAULT_GRID_PARAMS -- see
# comparative/trans_param_compare.py's PARAM_ALIASES for what's available.
GRID_PARAMS = DEFAULT_GRID_PARAMS
print('Datasets available:', list(ALL_DATASETS))
print(f'Comparing {SPEC_X.name} (x) vs {SPEC_Y.name} (y)')
print(f'{SPEC_X.name} cis genes with completed fit_trans: {SPEC_X.cis_genes}')
print(f'{SPEC_Y.name} cis genes with completed fit_trans: {SPEC_Y.cis_genes}')

## Single cis gene, quick look

Start here with one cis gene before looping over all shared ones below.

In [ ]:
CIS_GENE = 'GFI1B'

df_x = load_trans_summary(SPEC_X, CIS_GENE)
df_y = load_trans_summary(SPEC_Y, CIS_GENE)
merged = merge_pair(df_x, df_y, SPEC_X.name, SPEC_Y.name)
print(f'{SPEC_X.name}: {len(df_x)} trans genes | {SPEC_Y.name}: {len(df_y)} trans genes | shared: {len(merged)}')
merged.head()

### The starting-point plot: observed log2FC

In [ ]:
fig, ax = plt.subplots(figsize=(4.6, 4.6))
plot_obs_log2fc(merged, SPEC_X.name, SPEC_Y.name, CIS_GENE, ax=ax)
fig.savefig(os.path.join(PLOT_DIR, f'{CIS_GENE}_{SPEC_X.name}_vs_{SPEC_Y.name}_obs_log2fc.png'),
            dpi=150, bbox_inches='tight')
plt.show()

### Other parameters, colored + sorted by observed log2FC

In [ ]:
fig = plot_param_grid(merged, SPEC_X.name, SPEC_Y.name, CIS_GENE,
                       params=GRID_PARAMS, color_by_param='observed_log2fc', color_dataset=SPEC_X.name)
fig.savefig(os.path.join(PLOT_DIR, f'{CIS_GENE}_{SPEC_X.name}_vs_{SPEC_Y.name}_param_grid.png'),
            dpi=150, bbox_inches='tight')
plt.show()

## All shared cis genes

Loops the above over every cis gene present (with a completed fit_trans) in both datasets, saving both plots per gene to `PLOT_DIR`.

In [ ]:
results = compare_all_shared_cis_genes(
    SPEC_X, SPEC_Y,
    out_dir=PLOT_DIR,
    params=GRID_PARAMS,
    color_by_param='observed_log2fc',
)
print(f"Done. Wrote plots for: {list(results)}")

## Combine across cis genes

`results` holds one merged DataFrame per cis gene (columns suffixed `_{SPEC_X.name}` / `_{SPEC_Y.name}`) -- concatenate for an across-cis-gene view, e.g. overall correlation or a per-cis-gene breakdown.

In [ ]:
combined = pd.concat(
    [df.assign(cis_gene=gene) for gene, df in results.items()],
    ignore_index=True,
)

fig, ax = plt.subplots(figsize=(5, 5))
for gene, sub in combined.groupby('cis_gene'):
    ax.scatter(sub[f'observed_log2fc_{SPEC_X.name}'], sub[f'observed_log2fc_{SPEC_Y.name}'],
               s=10, alpha=0.5, label=gene)
ax.axhline(0, color='#ccc', lw=0.8)
ax.axvline(0, color='#ccc', lw=0.8)
ax.set_xlabel(f'observed log2FC ({SPEC_X.name})')
ax.set_ylabel(f'observed log2FC ({SPEC_Y.name})')
ax.legend(frameon=False, fontsize=8, title='cis gene')
ax.set_title(f'All cis genes: {SPEC_X.name} vs {SPEC_Y.name}')
fig.savefig(os.path.join(PLOT_DIR, f'ALL_{SPEC_X.name}_vs_{SPEC_Y.name}_obs_log2fc.png'),
            dpi=150, bbox_inches='tight')
plt.show()

## Customising further

- Swap `SPEC_X`/`SPEC_Y` for any pair in `comparative.datasets.ALL_DATASETS` (or add a new `DatasetSpec` there for a new run).
- `plot_param_grid(..., color_by_param=None)` turns off coloring (plain gray points).
- `plot_param_grid(..., color_dataset=SPEC_Y.name)` colors/sorts by the *other* dataset's observed log2FC instead.
- Pass `highlight_col=None` to any of the plotting functions to drop the `is_dependent` outline markers.
- `comparative/trans_param_compare.py`'s `PARAM_ALIASES` dict lists every parameter name recognised by `scatter_param`/`plot_param_grid` -- add an entry there for any column you want to compare that isn't already covered.